---
## 実験: スペクトルをbin幅10でビニングした特徴量でのモデル

In [51]:
import pandas as pd
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow import keras
import os

In [52]:
# ===== データ読み込み =====
df = pd.read_csv("data/middle/spectrum_bin_mean.csv", encoding='utf-8-sig')

meta_cols = ['sample number', 'species number', '樹種', '含水率']
feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].values
y = df['含水率'].values

In [53]:
# ===== 前処理 =====
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

input_dim = X_train_sc.shape[1]
print(f"特徴量数: {input_dim} | 訓練: {len(X_train)} | 検証: {len(X_val)}")



特徴量数: 600 | 訓練: 1057 | 検証: 265


In [63]:
# ===== チューニング パラメータ =====
hidden1_candidates = [64,96]   # 中間層1のノード数候補
hidden2_candidates = [128,140,160]    # 中間層2のノード数候補
# ===== 学習パラメータ =====
epochs_max  = 1000
batch_size  = 16
patience    = 100


In [64]:
# モデル構築関数
def build_model(h1, h2):
    model = keras.Sequential([
        keras.layers.Dense(h1, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dense(h2, activation='relu'),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [65]:
# パラメータチューニング
print(f"\n== チューニング開始: {len(hidden1_candidates) * len(hidden2_candidates)} 組 ==")
results = []
for h1, h2 in itertools.product(hidden1_candidates, hidden2_candidates):
    tf.random.set_seed(42)
    model = build_model(h1, h2)
    model.fit(
        X_train_sc, y_train,
        validation_data=(X_val_sc, y_val),
        epochs=epochs_max,
        batch_size=batch_size,
        callbacks=[keras.callbacks.EarlyStopping(patience=patience, restore_best_weights=True)],
        verbose=0
    )
    val_rmse = np.sqrt(mean_squared_error(y_val, model.predict(X_val_sc, verbose=0).flatten()))
    results.append({'hidden1': h1, 'hidden2': h2, 'val_rmse': val_rmse})
    print(f"  h1={h1:4d}, h2={h2:4d} -> val RMSE: {val_rmse:.4f}")




== チューニング開始: 6 組 ==
  h1=  64, h2= 128 -> val RMSE: 3.8242
  h1=  64, h2= 140 -> val RMSE: 3.4681
  h1=  64, h2= 160 -> val RMSE: 4.4153
  h1=  96, h2= 128 -> val RMSE: 3.1060
  h1=  96, h2= 140 -> val RMSE: 3.3087
  h1=  96, h2= 160 -> val RMSE: 3.3023


In [66]:
# ===== 結果表示 =====
results_df = pd.DataFrame(results).sort_values('val_rmse').reset_index(drop=True)
print("\n=== チューニング結果（昇順）===")
print(results_df.to_string())

best = results_df.iloc[0]
best_h1, best_h2 = int(best['hidden1']), int(best['hidden2'])
print(f"\n最良パラメータ: hidden1={best_h1}, hidden2={best_h2}, val RMSE={best['val_rmse']:.4f}")



=== チューニング結果（昇順）===
   hidden1  hidden2  val_rmse
0       96      128  3.106014
1       96      160  3.302302
2       96      140  3.308667
3       64      140  3.468112
4       64      128  3.824233
5       64      160  4.415347

最良パラメータ: hidden1=96, hidden2=128, val RMSE=3.1060


In [67]:

# ===== 最良パラメータで再学習（全訓練データ）=====
print("\n== ベストパラメータで再学習 ==")
X_all_sc = scaler.fit_transform(X)
tf.random.set_seed(42)
best_model = build_model(best_h1, best_h2)
best_model.summary()
best_model.fit(
    X_all_sc, y,
    epochs=epochs_max,
    batch_size=batch_size,
    callbacks=[keras.callbacks.EarlyStopping(patience=patience, restore_best_weights=True)],
    verbose=1
)


== ベストパラメータで再学習 ==
Model: "sequential_56"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_168 (Dense)           (None, 96)                57696     
                                                                 
 dense_169 (Dense)           (None, 128)               12416     
                                                                 
 dense_170 (Dense)           (None, 1)                 129       
                                                                 
Total params: 70241 (274.38 KB)
Trainable params: 70241 (274.38 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/1000
83/83 [==============================] - 0s 878us/step - loss: 2441.8083
Epoch 2/1000
83/83 [==============================] - 0s 829us/step - loss: 1845.2385
Epoch 3/1000
83/83 [==============================] - 0s 817us/step - loss: 1481.8646
E

In [68]:
# 最終的な訓練データに対するRMSEを計算
train_rmse = np.sqrt(mean_squared_error(y, best_model.predict(X_all_sc, verbose=0).flatten()))
print(f"\n最終 train RMSE: {train_rmse:.4f}")



最終 train RMSE: 3.6123


### 予測値の作成と保存

In [69]:
# ===== テストデータのビニング（trainと同じ処理）=====
test_raw = pd.read_csv("data/raw/test.csv", encoding='cp932')

test_meta_cols = ['sample number', 'species number', '樹種']
spec_cols_test = [c for c in test_raw.columns if c not in test_meta_cols]
spec_cols_float_test = [(c, float(c)) for c in spec_cols_test if float(c) >= 4000.0]

bins = np.arange(4000, 10010, 10)
bin_col_map = {}
for i in range(len(bins) - 1):
    low, high = bins[i], bins[i + 1]
    label = f"{int(low)}-{int(high)}"
    mask = [(c, v) for c, v in spec_cols_float_test if low <= v < high]
    if mask:
        bin_col_map[label] = [c for c, _ in mask]

test_bin = test_raw[test_meta_cols].copy()
for label, cols in bin_col_map.items():
    test_bin[label] = test_raw[cols].mean(axis=1)

C:\Users\keisu\AppData\Local\Temp\ipykernel_27204\1228241630.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_bin[label] = test_raw[cols].mean(axis=1)
C:\Users\keisu\AppData\Local\Temp\ipykernel_27204\1228241630.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_bin[label] = test_raw[cols].mean(axis=1)
C:\Users\keisu\AppData\Local\Temp\ipykernel_27204\1228241630.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

In [70]:
# ===== 予測 =====
feature_cols_order = [c for c in test_bin.columns if c not in test_meta_cols]
X_test = test_bin[feature_cols_order].values
X_test_sc = scaler.transform(X_test)  # trainで fitしたscalerをそのまま使用

preds = best_model.predict(X_test_sc, verbose=0).flatten()

In [75]:
# ===== 提出ファイル保存 =====
save_dir = "data/submission/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "koyama_experiment_neuralnet_v1.csv")

submit = pd.DataFrame({
    'sample_number': test_raw['sample number'].values,
    'pred': preds
})
submit.to_csv(save_path, index=False, header=False)
print(f"保存完了: {save_path}")
print(submit.head(10))

保存完了: data/submission/koyama_experiment_neuralnet_v1.csv
   sample_number        pred
0             95  170.262283
1             96  150.189209
2             97  170.132050
3             98  144.974976
4             99  142.691040
5            100  140.143433
6            101  135.567490
7            102  130.520706
8            103  128.930832
9            104  130.031525
